# The massive Endpoint test for debugging while serving

In [ ]:
HOST = 'http://0.0.0.0'
PORT = '8765'
URL  = '%s:%s' % (HOST, PORT)

In [ ]:
import json
import urllib.request


class RemoteEndpoint:
	def __init__(self, base_url = URL):
		self.base_url = base_url

		self.capabilities = self.get_capabilities()
		self.functions	  = self.get_functions()


	def get_capabilities(self):
		with urllib.request.urlopen('%s/meta' % self.base_url) as response:
			meta = json.load(response)

		capabilities = meta.get('capabilities')

		if capabilities is None:
			raise RuntimeError('The Endpoint metadata does not expose capabilities.')

		return capabilities


	def get_functions(self):
		functions = {}

		for cap in self.capabilities:
			assert cap['type'] == 'function'

			key = cap['function']['name']
			dsc = cap['function']['description']

			assert cap['function']['parameters']['type'] == 'object'

			req = cap['function']['parameters'].get('required', [])

			args = {}

			for nam, val in cap['function']['parameters']['properties'].items():
				args[nam] = {'typ': val['type'], 'dsc': val.get('description', ''), 'req': nam in req}

			functions[key] = {'dsc': dsc, 'args': args}

		return functions


	def run(self, fun_name, args, easy = True):
		fun = self.functions[fun_name]

		if easy:	# Help the caller by trying to match the provided arguments to the function's expected arguments.

			if type(args) is str and len(fun['args']) == 1:
				key, val = next(iter(fun['args'].items()))
				if val['typ'] == 'string':
					args = {key: args}

		data = json.dumps({'name': fun_name, 'arguments': args}).encode('utf-8')

		req = urllib.request.Request('%s/run' % self.base_url, data = data, headers = {'content-type': 'application/json'})

		with urllib.request.urlopen(req) as response:
			result = json.load(response)

		return result


In [ ]:
ep = RemoteEndpoint(URL)

In [ ]:
ep.functions

In [ ]:
ep.capabilities

In [ ]:
ep.run('get_items_by_index_bertrand_russell_works', '')

In [ ]:
get_functions()

In [ ]:
caps[0]['function']['parameters']

In [ ]:
cap = caps[0]

In [ ]:
for arg, value in cap['function']['parameters']['properties'].items():
	print(arg, value)

In [ ]:


def request_arguments(function_key, args):
	"""Converts convenient arguments into the Endpoint function-call format.

	Args:
		function_key (str): Name of an exposed function in functions.
		args (dict or object): A dictionary, or a value for a function with one parameter.

	Returns:
		(dict): Arguments ready for an Endpoint function-call request.
	"""

	if type(args) == dict:
		return args

	capability = functions.get(function_key)
	if capability is None:
		raise KeyError('Function %s is not exposed by the Endpoint.' % function_key)

	properties = capability['function']['parameters'].get('properties', {})
	if len(properties) != 1:
		raise ValueError('Function %s has %d parameters; pass args as a dictionary.' % (function_key, len(properties)))

	argument_name = next(iter(properties))
	return {argument_name: args}


def endpoint_call(path, function_key, args, timeout = 10):
	"""Sends a function-call request to an Endpoint route.

	Args:
		path (str): Endpoint route, either 'run' or 'dry_run'.
		function_key (str): Name of the exposed Endpoint function to call.
		args (dict or object): Function arguments in convenient form.
		timeout (int): Maximum number of seconds to wait for the response.

	Returns:
		(object): JSON result returned by the Endpoint.
	"""

	payload = {'name': function_key, 'arguments': request_arguments(function_key, args)}
	data = json.dumps(payload).encode('utf-8')
	url = '%s/%s' % (URL.rstrip('/'), path)
	request = urllib.request.Request(url, data = data, headers = {'Content-Type': 'application/json'}, method = 'POST')

	with urllib.request.urlopen(request, timeout = timeout) as response:
		return json.load(response)


def run(function_key, args, timeout = 10):
	"""Runs an exposed Endpoint function.

	Args:
		function_key (str): Name of the exposed Endpoint function to call.
		args (dict or object): Function arguments in convenient form.
		timeout (int): Maximum number of seconds to wait for the response.

	Returns:
		(object): JSON result returned by the Endpoint.
	"""

	return endpoint_call('run', function_key, args, timeout)


def dry_run(function_key, args, timeout = 10):
	"""Checks an exposed Endpoint function call without running it.

	Args:
		function_key (str): Name of the exposed Endpoint function to check.
		args (dict or object): Function arguments in convenient form.
		timeout (int): Maximum number of seconds to wait for the response.

	Returns:
		(object): JSON result returned by the Endpoint.
	"""

	return endpoint_call('dry_run', function_key, args, timeout)

In [ ]:
functions = get_functions()
functions